In [1]:
import mcstasscript as ms
import mcstastox as mx
import scipp as sc
from scipp.typing import VariableLike
import scipp as sc
from scippneutron.conversion.graph.beamline import beamline
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import plopp as pp

parent = os.path.dirname(os.getcwd())
sys.path.append(parent)

from trex_reduction import inelastic, produce_trex_event_object, _calc_pulse_centroid

In [2]:
file_path = parent + "/runs/LET_vanad"

with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp(
        source_name="SourceMantid",
        sample_name="iso_samp",
    )

data = ms.load_data(file_path)

In [4]:
# Load event data into scipp
event_object = scipp_object
# McStas provides absolute time, not time of flight
event_object["events"].bins.coords["tof"] = event_object["events"].bins.coords["t"]
# Add additional information required for inelastic scattering
event_object["events"] = produce_trex_event_object(
    event_object["events"], file_path, "Monitor6"
)

qens_graph = {**beamline(scatter=True), **inelastic}
event_object["events"] = event_object["events"].transform_coords(["Q","mag_Q","dE"], graph=qens_graph)
event_object["events"]

<scipp.DataArray>
Dimensions: Sizes[pixel_id:25886, ]
Coordinates:
  L1                        float64              [m]  ()  25
  L2                        float64              [m]  (pixel_id)  [4.02385, 4.02385, ..., 4.02385, 4.02385]
  Lm                        float64              [m]  ()  23.505
  incident_beam             vector3              [m]  ()  (0, 0, 25)
  monitor_beam              vector3              [m]  ()  (0, 0, 23.505)
* monitor_counts            float64  [dimensionless]  ()  1.65719e+06
  monitor_position          vector3              [m]  ()  (0, 0, 23.505)
* pixel_id                    int64  [dimensionless]  (pixel_id)  [0, 1, ..., 30053, 30054]
  position                  vector3              [m]  (pixel_id)  [(-2.23064, -1.98529, 27.6971), (-2.19208, -1.98529, 27.7285), ..., (2.3436, 1.98529, 22.4005), (2.30641, 1.98529, 22.3674)]
  sample_position           vector3              [m]  ()  (0, 0, 25)
  scattered_beam            vector3              [m]  (pixel_id)  [(-2.23064, -1.98529, 2.69708), (-2.19208, -1.98529, 2.72851), ..., (2.3436, 1.98529, -2.59953), (2.30641, 1.98529, -2.63258)]
  source_position           vector3              [m]  ()  (0, 0, 0)
  unit_kf                   vector3  [dimensionless]  (pixel_id)  [(-0.554355, -0.493381, 0.670273), (-0.544771, -0.493381, 0.678085), ..., (0.582427, 0.493381, -0.646029), (0.573185, 0.493381, -0.654243)]
  unit_ki                   vector3  [dimensionless]  ()  (0, 0, 1)
Data:
                          DataArrayView        <no unit>  (pixel_id)  binned data: dim='events', content=DataArray(
          dims=(events: 59921),
          data=float64[counts],
          coords={'t':float64[s], 'tof':float64[s], 'time_on_monitor':float64[s],
                  'vi':float64[m/s], 'mag_ki':float64[1/Å], 'time_on_sample':float64[s],
                  'ki':vector3[1/Å], 'vf':float64[m/s], 'mag_kf':float64[1/Å],
                  'dE':float64[meV], 'kf':vector3[1/Å], 'Q':vector3[1/Å],
                  'mag_Q':float64[1/Å]})

In [6]:
event_object["events"].bins.coords["qx"] = (
    event_object["events"].bins.coords["Q"].fields["x"]
)
event_object["events"].bins.coords["qy"] = (
    event_object["events"].bins.coords["Q"].fields["y"]
)
event_object["events"].bins.coords["qz"] = (
    event_object["events"].bins.coords["Q"].fields["z"]
)
event_object["events"].bins.coords["en"] = event_object["events"].bins.coords["dE"]

# Calculate minimum and maximum kf

In [7]:
event_object["events"].bins.coords

<scipp.Dict>
  t: <scipp.Variable> (events: 59921)    float64              [s]  [0.0345534, 0.0345478, ..., 0.0345157, 0.0346881]
  tof: <scipp.Variable> (events: 59921)    float64              [s]  [0.0345534, 0.0345478, ..., 0.0345157, 0.0346881]
  time_on_monitor: <scipp.Variable> (events: 59921)    float64              [s]  [0.0280169, 0.0280169, ..., 0.0280169, 0.0280169]
  vi: <scipp.Variable> (events: 59921)    float64            [m/s]  [838.957, 838.957, ..., 838.957, 838.957]
  mag_ki: <scipp.Variable> (events: 59921)    float64           [1/Å]  [1.33248, 1.33248, ..., 1.33248, 1.33248]
  time_on_sample: <scipp.Variable> (events: 59921)    float64              [s]  [0.0297989, 0.0297989, ..., 0.0297989, 0.0297989]
  ki: <scipp.Variable> (events: 59921)    vector3           [1/Å]  [(0, 0, 1.33248), (0, 0, 1.33248), ..., (0, 0, 1.33248), (0, 0, 1.33248)]
  vf: <scipp.Variable> (events: 59921)    float64            [m/s]  [846.323, 847.33, ..., 853.087, 823.01]
  mag_kf: <scipp.V

In [8]:
from scipy.ndimage import label

Lm = event_object["events"].coords["Lm"]


def _calc_mag_kf_from_ef(ef):
    hbar = sc.constants.hbar
    m_n = sc.constants.neutron_mass
    kf = sc.sqrt(2 * m_n * ef) / hbar
    return kf.to(unit="1/angstrom")


monitor = ms.name_search("Monitor6", data)
tom_centroid = _calc_pulse_centroid(monitor)
vi = Lm / tom_centroid

ei = (0.5 * sc.constants.m_n * vi**2).to(unit="meV")

unit_ki = sc.vector([0, 0, 1])
mag_ki = ((sc.constants.neutron_mass * vi) / sc.constants.hbar).to(unit="1/angstrom")

ki = unit_ki * mag_ki


prop_ei = 0.8
max_ef = (1 + prop_ei) * ei
min_ef = (1 - prop_ei) * ei

min_kf = _calc_mag_kf_from_ef(min_ef)
max_kf = _calc_mag_kf_from_ef(max_ef)

print(f"min_kf: {min_kf}, max_kf: {max_kf}")

min_kf: <scipp.Variable> (tof: 1)    float64            [1/Å]  [0.595902], max_kf: <scipp.Variable> (tof: 1)    float64            [1/Å]  [1.78771]


# Access and calculate detector trajectory endpoints

tan(theta) = x_vec / z_vec

In [10]:
sample_position = event_object["events"].coords["sample_position"]
pixel_vec = event_object["positions"] - sample_position
pixel_vec = pixel_vec / sc.norm(pixel_vec)

Q_max = ki.to(unit="1/Å") - (pixel_vec * max_kf)
Q_min = ki.to(unit="1/Å") - (pixel_vec * min_kf)

kf_max = sc.broadcast(max_kf, dims=Q_max.dims, shape=Q_max.shape)
kf_min = sc.broadcast(min_kf, dims=Q_min.dims, shape=Q_min.shape)


In [11]:
import scipp as sc


def _make_bins(spec, dim, unit="1/Å"):
    if not isinstance(spec, tuple):
        raise TypeError(f"{dim} must be a tuple of length 2 or 3")

    if len(spec) == 2:
        start, stop = spec
        step = stop - start

    elif len(spec) == 3:
        start, stop, step = spec

    else:
        raise ValueError(f"{dim} tuple must have length 2 or 3")

    return sc.arange(dim=dim, start=start, stop=stop + step, step=step, unit=unit)


def generate_bins(qx=None, qy=None, qz=None, en=None):
    """
    Generate Scipp bin edges for x, y, z, and en.

    Each argument must be:
      - (min, max)
      - (start, stop, step)

    Returns
    -------
    dict[str, sc.Variable]
        Bin edges per dimension
    """
    bins = {}

    if qx is not None:
        bins["qx"] = _make_bins(qx, "qx")

    if qy is not None:
        bins["qy"] = _make_bins(qy, "qy")

    if qz is not None:
        bins["qz"] = _make_bins(qz, "qz")

    if en is not None:
        bins["en"] = _make_bins(en, "en", unit="meV")

    return bins

In [12]:
bins = generate_bins(
    qx=(-2, 1.5, 0.1), qy=(-0.1, 0.1), qz=(-1, 3.0, 0.1), en=(-0.2, 0.2)
)

In [13]:
hbar = sc.constants.hbar
m_n = sc.constants.neutron_mass

kf_array = (
    (sc.sqrt(2 * m_n * (ei[0] - bins["en"])) / hbar)
    .to(unit="1/Å")
    .rename_dims({"en": "mag_kf"})
)
bins["mag_kf"] = kf_array[~sc.isnan(kf_array)]
bins["mag_kf"] = sc.sort(bins["mag_kf"], key="mag_kf")

In [14]:
import numpy as np


def clip_segment_to_box_4d(p0, p1, edges, eps=1e-12):
    p0 = np.asarray(p0, float)
    p1 = np.asarray(p1, float)
    d = p1 - p0

    box_min = np.array([e[0] for e in edges])
    box_max = np.array([e[-1] for e in edges])

    t0, t1 = 0.0, 1.0

    for i in range(4):
        if abs(d[i]) < eps:
            if p0[i] < box_min[i] or p0[i] > box_max[i]:
                return None
            continue

        inv = 1.0 / d[i]
        ta = (box_min[i] - p0[i]) * inv
        tb = (box_max[i] - p0[i]) * inv

        if ta > tb:
            ta, tb = tb, ta

        t0 = max(t0, ta)
        t1 = min(t1, tb)

        if t0 > t1:
            return None

    return t0, t1


def voxel_traversal_4d(p0, p1, edges, eps=1e-12):
    """
    Yields for each crossed voxel:
    (index_tuple, entry_point, exit_point)
    """

    p0 = np.asarray(p0, float)
    p1 = np.asarray(p1, float)
    d = p1 - p0

    edges = [np.asarray(e, float) for e in edges]

    clipped = clip_segment_to_box_4d(p0, p1, edges, eps)
    if clipped is None:
        return

    t_enter, t_exit = clipped

    # ---- Ensure starting voxel is included ----
    p_start = p0 + (t_enter + eps) * d

    # Initial voxel index
    idx = np.array(
        [np.searchsorted(edges[i], p_start[i], side="right") - 1 for i in range(4)],
        dtype=int,
    )

    step = np.sign(d).astype(int)

    # Compute next boundary crossings
    tNext = np.zeros(4)

    for i in range(4):
        if abs(d[i]) < eps:
            tNext[i] = np.inf
            continue

        if step[i] > 0:
            boundary = edges[i][idx[i] + 1]
        else:
            boundary = edges[i][idx[i]]

        tNext[i] = (boundary - p0[i]) / d[i]

    t = t_enter

    while (
        np.all(idx >= 0)
        and all(idx[i] < len(edges[i]) - 1 for i in range(4))
        and t <= t_exit + eps
    ):
        axis = np.argmin(tNext)
        t_voxel_exit = min(tNext[axis], t_exit)

        entry = p0 + t * d
        exit_ = p0 + t_voxel_exit * d

        yield tuple(idx), entry, exit_

        if tNext[axis] > t_exit:
            break

        # Step to next voxel
        idx[axis] += step[axis]
        t = tNext[axis]

        if idx[axis] < 0 or idx[axis] >= len(edges[axis]) - 1:
            break

        # Update next boundary crossing for this axis
        if step[axis] > 0:
            boundary = edges[axis][idx[axis] + 1]
        else:
            boundary = edges[axis][idx[axis]]

        tNext[axis] = (boundary - p0[axis]) / d[axis]

In [15]:
edges = {key: bins[key] for key in ("qx", "qy", "qz", "mag_kf")}
edges

{'qx': <scipp.Variable> (qx: 36)    float64           [1/Å]  [-2, -1.9, ..., 1.4, 1.5],
 'qy': <scipp.Variable> (qy: 2)    float64           [1/Å]  [-0.1, 0.1],
 'qz': <scipp.Variable> (qz: 41)    float64           [1/Å]  [-1, -0.9, ..., 2.9, 3],
 'mag_kf': <scipp.Variable> (mag_kf: 2)    float64           [1/Å]  [1.29575, 1.36822]}

In [16]:
%matplotlib widget
binned_counts = sc.bin(event_object["events"].data, **edges)
pp.plot(binned_counts.hist().squeeze().transpose(), coords=["qx", "qz"])

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

# Calculate dE 

In [ ]:
import numpy as np


def clip_segment_to_box_4d(p0, p1, edges, eps=1e-12):
    p0 = np.asarray(p0, float)
    p1 = np.asarray(p1, float)
    d = p1 - p0

    box_min = np.array([e[0] for e in edges])
    box_max = np.array([e[-1] for e in edges])

    t0, t1 = 0.0, 1.0

    for i in range(4):
        if abs(d[i]) < eps:
            if p0[i] < box_min[i] or p0[i] > box_max[i]:
                return None
            continue

        inv = 1.0 / d[i]
        ta = (box_min[i] - p0[i]) * inv
        tb = (box_max[i] - p0[i]) * inv

        if ta > tb:
            ta, tb = tb, ta

        t0 = max(t0, ta)
        t1 = min(t1, tb)

        if t0 > t1:
            return None

    return t0, t1


def voxel_traversal_4d(p0, p1, edges, eps=1e-12):
    """
    Yields for each crossed voxel:
    (index_tuple, entry_point, exit_point)
    """

    p0 = np.asarray(p0, float)
    p1 = np.asarray(p1, float)
    d = p1 - p0

    edges = [np.asarray(e, float) for e in edges]

    clipped = clip_segment_to_box_4d(p0, p1, edges, eps)
    if clipped is None:
        return

    t_enter, t_exit = clipped

    # ---- Ensure starting voxel is included ----
    p_start = p0 + (t_enter + eps) * d

    # Initial voxel index
    idx = np.array(
        [np.searchsorted(edges[i], p_start[i], side="right") - 1 for i in range(4)],
        dtype=int,
    )

    step = np.sign(d).astype(int)

    # Compute next boundary crossings
    tNext = np.zeros(4)

    for i in range(4):
        if abs(d[i]) < eps:
            tNext[i] = np.inf
            continue

        if step[i] > 0:
            boundary = edges[i][idx[i] + 1]
        else:
            boundary = edges[i][idx[i]]

        tNext[i] = (boundary - p0[i]) / d[i]

    t = t_enter

    while (
        np.all(idx >= 0)
        and all(idx[i] < len(edges[i]) - 1 for i in range(4))
        and t <= t_exit + eps
    ):
        axis = np.argmin(tNext)
        t_voxel_exit = min(tNext[axis], t_exit)

        entry = p0 + t * d
        exit_ = p0 + t_voxel_exit * d

        yield tuple(idx), entry, exit_

        if tNext[axis] > t_exit:
            break

        # Step to next voxel
        idx[axis] += step[axis]
        t = tNext[axis]

        if idx[axis] < 0 or idx[axis] >= len(edges[axis]) - 1:
            break

        # Update next boundary crossing for this axis
        if step[axis] > 0:
            boundary = edges[axis][idx[axis] + 1]
        else:
            boundary = edges[axis][idx[axis]]

        tNext[axis] = (boundary - p0[axis]) / d[axis]

In [ ]:
edges

{'qx': <scipp.Variable> (qx: 36)    float64           [1/Å]  [-2, -1.9, ..., 1.4, 1.5],
 'qy': <scipp.Variable> (qy: 2)    float64           [1/Å]  [-0.1, 0.1],
 'qz': <scipp.Variable> (qz: 41)    float64           [1/Å]  [-1, -0.9, ..., 2.9, 3],
 'mag_kf': <scipp.Variable> (mag_kf: 2)    float64           [1/Å]  [1.29575, 1.36822]}